# ADS ViT Forensics — Colab Quickstart

End-to-end Colab workflow for generating the JSON artifacts and figures used in

**Attention Divergence Score: A Forensic Metric for Characterizing Parameter-Level Attacks in Vision Transformers**.

This notebook is intended for the GitHub repository. It assumes that the trained ViT-Base checkpoints are stored on Google Drive and that the repository scripts are available either by cloning the repo or by running the notebook from a checked-out copy.

The full pipeline is GPU-oriented and long-running. The verification step (`reproduce.py`) is CPU-only and can be run independently once the JSON files are present.


## 0. Runtime and repository setup

Use a GPU runtime for experiment generation. The recommended sequence is:

1. Mount Google Drive.
2. Clone or update the repository.
3. Install dependencies.
4. Edit the path configuration cell.
5. Run the experiment sections needed for your update.


In [ ]:
# GPU sanity check
import os
import sys
import subprocess
from pathlib import Path

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("PyTorch not available yet:", exc)

# Also show nvidia-smi when available.
subprocess.run("nvidia-smi || true", shell=True, check=False)


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# Clone or update the repository.
# If you uploaded/edited scripts manually in /content, set REPO_DIR accordingly instead.
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/djokobandjur/ads-vit-forensics.git"
REPO_DIR = Path("/content/ads-vit-forensics")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
print("Current files:")
subprocess.run(["ls", "-lah"], check=False)


In [ ]:
# Install repository dependencies.
# This can take a few minutes on a fresh Colab runtime.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
print("Dependencies installed.")


## 1. Path configuration

Edit these paths once. The defaults mirror the Drive layout used for the paper runs, but the notebook is path-configurable.


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

# Output location for this paper run.
RUN_DIR = DRIVE_ROOT / "ads_tifs_final_n6"
DATA_DIR = RUN_DIR / "data"
OUTPUT_DIR = RUN_DIR / "output"
LOG_DIR = RUN_DIR / "logs"

# Trained model checkpoint folders.
IMAGENET_MODELS_DIR = DRIVE_ROOT / "pe_experiment/results"
CIFAR_MODELS_DIR = DRIVE_ROOT / "pe_experiment/results_cifar100"

# ImageNet-100 validation data.
# The tar should expand to /content/imagenet100_resized/{train,val}.
IMAGENET100_TAR = DRIVE_ROOT / "pe_experiment/imagenet/imagenet100_resized.tar"
IMAGENET100_ROOT = Path("/content/imagenet100_resized")
IMAGENET100_VAL_DIR = IMAGENET100_ROOT / "val"

# CIFAR-100 cache. If the dataset is already on Drive, it can be copied here;
# otherwise torchvision can download it when the scripts run.
CIFAR100_CACHE = Path("/tmp/cifar100")
CIFAR100_DRIVE_SOURCE = DRIVE_ROOT / "cifar100_data/cifar-100-python"

# Final primary protocol seeds used by the current manuscript/repository artifacts.
PRIMARY_SEEDS = [42, 123, 456, 789, 1011, 1213]

for p in [DATA_DIR, OUTPUT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("LOG_DIR:", LOG_DIR)
print("ImageNet models:", IMAGENET_MODELS_DIR)
print("CIFAR models:", CIFAR_MODELS_DIR)
print("ImageNet val:", IMAGENET100_VAL_DIR)
print("CIFAR cache:", CIFAR100_CACHE)


## 2. Helper functions

The helpers below run each script with logging to Drive. They do not hide errors: if a script exits with a non-zero code, the cell fails.


In [ ]:
import shlex
import subprocess
from datetime import datetime


def run_logged(args, log_name, cwd=REPO_DIR):
    """Run a command and tee stdout/stderr to a log file in LOG_DIR."""
    args = [str(a) for a in args]
    log_path = LOG_DIR / log_name
    print("\n$", " ".join(shlex.quote(a) for a in args))
    print("Log:", log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "w", encoding="utf-8") as log:
        log.write(f"# Started: {datetime.now().isoformat()}\n")
        log.write("$ " + " ".join(shlex.quote(a) for a in args) + "\n\n")
        proc = subprocess.Popen(
            args,
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        ret = proc.wait()
        log.write(f"\n# Finished: {datetime.now().isoformat()}  returncode={ret}\n")
    if ret != 0:
        raise RuntimeError(f"Command failed with return code {ret}: {' '.join(args)}")
    return log_path


def py_script(script_name, *script_args, log_name=None):
    """Run a Python script from the repository root."""
    if log_name is None:
        log_name = script_name.replace('.py', '.log')
    return run_logged([sys.executable, "-u", str(REPO_DIR / script_name), *script_args], log_name)


## 3. Dataset preparation

ImageNet-100 is not redistributed. This cell extracts the private ImageNet-100 tarball from Drive if the validation directory is not already present.

CIFAR-100 can either be copied from Drive or downloaded by torchvision scripts.


In [ ]:
import shutil

# ImageNet-100 extraction.
if IMAGENET100_VAL_DIR.exists():
    print("ImageNet-100 validation directory already exists:", IMAGENET100_VAL_DIR)
else:
    if not IMAGENET100_TAR.exists():
        raise FileNotFoundError(
            f"ImageNet-100 tar not found: {IMAGENET100_TAR}\n"
            "Edit IMAGENET100_TAR in the configuration cell or prepare the dataset manually."
        )
    print("Extracting:", IMAGENET100_TAR)
    shutil.rmtree(IMAGENET100_ROOT, ignore_errors=True)
    run_logged(["tar", "-C", "/content", "-xf", str(IMAGENET100_TAR)], "extract_imagenet100.log", cwd=Path("/content"))

# Basic validation of ImageNet-100 folder layout.
if IMAGENET100_VAL_DIR.exists():
    class_dirs = [p for p in IMAGENET100_VAL_DIR.iterdir() if p.is_dir()]
    print("ImageNet-100 val classes:", len(class_dirs))
    print("ImageNet-100 val dir:", IMAGENET100_VAL_DIR)
else:
    print("ImageNet-100 validation directory not found yet.")

# CIFAR-100 cache setup.
CIFAR100_CACHE.mkdir(parents=True, exist_ok=True)
if CIFAR100_DRIVE_SOURCE.exists():
    target = CIFAR100_CACHE / "cifar-100-python"
    if not target.exists():
        print("Copying CIFAR-100 from Drive to local cache:", target)
        shutil.copytree(CIFAR100_DRIVE_SOURCE, target)
    else:
        print("CIFAR-100 local cache already exists:", target)
else:
    print("CIFAR-100 Drive source not found; torchvision scripts may download it automatically.")


## 4. Reference indices

The paper uses a fixed 256-image ImageNet-100 reference set. The shipped `data/ads_ref_indices.json` should be copied into the run data folder for exact reproducibility.


In [ ]:
import json
import shutil

repo_ref = REPO_DIR / "data" / "ads_ref_indices.json"
run_ref = DATA_DIR / "ads_ref_indices.json"

if repo_ref.exists() and not run_ref.exists():
    shutil.copy2(repo_ref, run_ref)
    print("Copied reference indices to:", run_ref)
elif run_ref.exists():
    print("Reference indices already present:", run_ref)
else:
    print("Reference indices not found in repo data/. Scripts may regenerate them with their fixed seed.")

if run_ref.exists():
    ref = json.loads(run_ref.read_text())
    print("Reference count:", len(ref))
    print("Unique:", len(set(ref)) == len(ref))
    print("Min/max:", min(ref), max(ref))


## 5. Main PE-only ADS sweeps

Outputs:

- `ads_results.json`
- `ads_results_cifar100.json`


In [ ]:
py_script(
    "ads_experiment.py",
    "--models_dir", IMAGENET_MODELS_DIR,
    "--val_dir", IMAGENET100_VAL_DIR,
    "--output_path", DATA_DIR / "ads_results.json",
    log_name="ads_experiment_imagenet.log",
)


In [ ]:
py_script(
    "ads_experiment_cifar.py",
    "--models_dir", CIFAR_MODELS_DIR,
    "--val_dir", CIFAR100_CACHE,
    "--output_path", DATA_DIR / "ads_results_cifar100.json",
    log_name="ads_experiment_cifar100.log",
)


## 6. Specificity sweeps: PE, QKV, MLP, all-weights

Outputs:

- `ads_specificity.json`
- `ads_specificity_cifar.json`


In [ ]:
py_script(
    "ads_specificity.py",
    "--models_dir", IMAGENET_MODELS_DIR,
    "--val_dir", IMAGENET100_VAL_DIR,
    "--output_path", DATA_DIR / "ads_specificity.json",
    log_name="ads_specificity_imagenet.log",
)


In [ ]:
py_script(
    "ads_specificity_cifar.py",
    "--models_dir", CIFAR_MODELS_DIR,
    "--val_dir", CIFAR100_CACHE,
    "--output_path", DATA_DIR / "ads_specificity_cifar.json",
    log_name="ads_specificity_cifar100.log",
)


## 7. ROC analysis against benign shifts

Output:

- `ads_roc_v2.json`


In [ ]:
py_script(
    "ads_roc_analysis.py",
    "--models_dir", IMAGENET_MODELS_DIR,
    "--val_dir", IMAGENET100_VAL_DIR,
    "--output_path", DATA_DIR / "ads_roc_v2.json",
    log_name="ads_roc_analysis.log",
)


## 8. Adaptive and reference-set evasion analyses

Outputs:

- `ads_adaptive.json`
- `ads_ref_evasion.json`


In [ ]:
py_script(
    "ads_adaptive.py",
    "--models_dir", IMAGENET_MODELS_DIR,
    "--val_dir", IMAGENET100_VAL_DIR,
    "--output_path", DATA_DIR / "ads_adaptive.json",
    log_name="ads_adaptive.log",
)


In [ ]:
py_script(
    "ads_ref_evasion.py",
    "--models_dir", IMAGENET_MODELS_DIR,
    "--val_dir", IMAGENET100_VAL_DIR,
    "--output_path", DATA_DIR / "ads_ref_evasion.json",
    log_name="ads_ref_evasion.log",
)


## 9. Fine-grid threshold calibration

Output:

- `ads_threshold_fine.json`


In [ ]:
py_script(
    "ads_threshold_fine.py",
    "--models_dir", IMAGENET_MODELS_DIR,
    "--val_dir", IMAGENET100_VAL_DIR,
    "--output_path", DATA_DIR / "ads_threshold_fine.json",
    log_name="ads_threshold_fine.log",
)


## 10. Residual-stream probing

Outputs:

- `ads_probing_residual.json`
- `ads_probing_residual_cifar.json`


In [ ]:
py_script(
    "ads_probing_residual.py",
    "--models_dir", IMAGENET_MODELS_DIR,
    "--val_dir", IMAGENET100_VAL_DIR,
    "--output_path", DATA_DIR / "ads_probing_residual.json",
    log_name="ads_probing_residual_imagenet.log",
)


In [ ]:
py_script(
    "ads_probing_residual_cifar.py",
    "--models_dir", CIFAR_MODELS_DIR,
    "--val_dir", CIFAR100_CACHE,
    "--output_path", DATA_DIR / "ads_probing_residual_cifar.json",
    log_name="ads_probing_residual_cifar100.log",
)


## 11. Detection-method comparison

Output:

- `ads_comparison.json`


In [ ]:
py_script(
    "ads_comparison.py",
    "--models_dir", IMAGENET_MODELS_DIR,
    "--val_dir", IMAGENET100_VAL_DIR,
    "--output_path", DATA_DIR / "ads_comparison.json",
    log_name="ads_comparison.log",
)


## 12. Generate paper figures

This uses the repository version of `generate_ads_figures.py`, including the Fig. 2 log-scale zero-column fix.

Outputs are written under `OUTPUT_DIR/figures/`.


In [ ]:
py_script(
    "generate_ads_figures.py",
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    log_name="generate_ads_figures.log",
)

fig_dir = OUTPUT_DIR / "figures"
print("\nGenerated figures:")
if fig_dir.exists():
    for p in sorted(fig_dir.glob("ads_fig*")):
        print(" ", p)
else:
    print("Figure directory not found:", fig_dir)


## 13. CPU-only verification

Run this after the JSON files are present. It does not require GPU or model checkpoints.


In [ ]:
py_script(
    "reproduce.py",
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    "--no-figures",
    log_name="reproduce.log",
)


## 14. Final artifact inventory and optional zip

Use this cell before uploading artifacts to GitHub/Zenodo or before comparing a new run to the shipped repository artifacts.


In [ ]:
import zipfile

print("JSON files:")
for p in sorted(DATA_DIR.glob("*.json")):
    print(f"  {p.name:36s} {p.stat().st_size/1024/1024:8.2f} MB")

print("\nFigure files:")
fig_dir = OUTPUT_DIR / "figures"
if fig_dir.exists():
    for p in sorted(fig_dir.glob("ads_fig*")):
        print(f"  {p.name:36s} {p.stat().st_size/1024/1024:8.2f} MB")

zip_path = RUN_DIR / "ads_tifs_final_n6_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [DATA_DIR, OUTPUT_DIR, LOG_DIR]:
        if folder.exists():
            for p in folder.rglob("*"):
                if p.is_file():
                    zf.write(p, p.relative_to(RUN_DIR))

print("\nCreated zip:", zip_path)


## Notes

- The full generation pipeline is expensive. For routine repository checks, use `reproduce.py` against the shipped JSON files.
- Keep `data/ads_ref_indices.json` fixed to reproduce the paper’s reference-set measurements exactly.
- The repository uses standard script names without the historical `_n6` suffix.
- Generated ImageNet data are not redistributed; only JSON results, figures, code, and fixed reference indices belong in the public repo.
